# Modélisation de données avec des algorithmes linéaires

Ce notebook a pour objectif d'explorer et de comparer plusieurs modèles de régression linéaire appliqués aux comptages de vélos à Paris.

L’objectif est de comprendre comment les techniques de régularisation peuvent influencer les résultats et rendre les modèles plus fiables.

Les algorithmes étudiés sont :

- **Régression linéaire** : simple, sans pénalisation.
- **Ridge** : introduit une pénalisation afin de réduire l’importance des coefficients trop élevés et limiter le surapprentissage.
- **Lasso** : peut annuler certaines variables, permettant une sélection automatique des variables les plus pertinentes.
- **Elastic Net** : un compromis entre Ridge et Lasso pour gérer à la fois la sélection de variables et la corrélation entre elles.

Les performances des modèles sont comparées à l’aide d’indicateurs tels que le **MAE**, le **RMSE** et le **R²**.

Cette approche constitue une étape importante avant de tester des modèles plus complexes (arbres de décision et gradient boosting), afin d’établir une base solide pour la modélisation prédictive.

## Import des librairies et des données

In [1]:
import joblib
import pandas as pd
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [2]:
df = pd.read_csv('../data/processed/df_processed.csv', sep=',')

In [3]:
df.head()

,Nom du compteur,Nom du site de comptage,Comptage horaire,Date et heure de comptage,Lien vers photo du site de comptage,Direction,Latitude,Longitude,Température (°C),Précipitations (mm),...,Mois,Année,Heure,Jour de la semaine,Week-end,Vacances,lag_1h,lag_24h,lag_168h,roll_mean_3h
0,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,0,2025-01-07 11:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,4.3,1.8,...,1,2025,11,1,0,0,NaN,NaN,NaN,NaN
1,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,13,2025-01-07 12:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,5.4,0.0,...,1,2025,12,1,0,0,0.0,NaN,NaN,NaN
2,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,50,2025-01-07 13:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.3,0.0,...,1,2025,13,1,0,0,13.0,NaN,NaN,NaN
3,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,54,2025-01-07 14:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.8,0.5,...,1,2025,14,1,0,0,50.0,NaN,NaN,21.0
4,10 avenue de la Grande Armée 10 avenue de la G...,10 avenue de la Grande Armée,33,2025-01-07 15:00:00,https://filer.eco-counter-tools.com/file/26/1d...,Bike IN,48.8748,2.2924,6.9,0.0,...,1,2025,15,1,0,0,54.0,NaN,NaN,39.0


## Préparation des données avant entraînement

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 947136 entries, 0 to 947135
Data columns (total 21 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Nom du compteur                      947136 non-null  object 
 1   Nom du site de comptage              947136 non-null  object 
 2   Comptage horaire                     947136 non-null  int64  
 3   Date et heure de comptage            947136 non-null  object 
 4   Lien vers photo du site de comptage  937306 non-null  object 
 5   Direction                            947136 non-null  object 
 6   Latitude                             947136 non-null  float64
 7   Longitude                            947136 non-null  float64
 8   Température (°C)                     947136 non-null  float64
 9   Précipitations (mm)                  947136 non-null  float64
 10  Jour du mois                         947136 non-null  int64  
 11  Mois         

In [5]:
df.columns

Index(['Nom du compteur', 'Nom du site de comptage', 'Comptage horaire',
       'Date et heure de comptage', 'Lien vers photo du site de comptage',
       'Direction', 'Latitude', 'Longitude', 'Température (°C)',
       'Précipitations (mm)', 'Jour du mois', 'Mois', 'Année', 'Heure',
       'Jour de la semaine', 'Week-end', 'Vacances', 'lag_1h', 'lag_24h',
       'lag_168h', 'roll_mean_3h'],
      dtype='object')

In [6]:
feature_cols_num = [
    'Année', 'Mois', 'Jour du mois', 'Heure', 'Jour de la semaine',
    'Week-end', 'Vacances', 'lag_1h', 'lag_24h',
    'lag_168h', 'roll_mean_3h', 'Température (°C)', 'Précipitations (mm)',
]

feature_cols_cat = ['Nom du compteur', 'Direction']

for col in feature_cols_num:
    df[col] = pd.to_numeric(df[col], errors='coerce')
# Assure que toutes les colonnes numériques sont bien en format numérique

df_clean = df.dropna(
    subset=feature_cols_num + feature_cols_cat + ['Comptage horaire']
).copy()
# Supprime les valeurs vides (en particulier celles des colonnes lag)

df['Date et heure de comptage'] = pd.to_datetime(
    df['Date et heure de comptage'], errors='coerce'
)
# Convertit 'Date et heure de comptage' en format datetime

df_clean = df_clean.sort_values('Date et heure de comptage')

split = int(len(df_clean) * 0.8)
train = df_clean.iloc[:split]
test = df_clean.iloc[split:]
# Sépare le dataset chronologiquement

X_train = train[feature_cols_num + feature_cols_cat]
y_train = train['Comptage horaire']
X_test = test[feature_cols_num + feature_cols_cat]
y_test = test['Comptage horaire']

# Création d'un pipeline : on standardise les variables numériques
# et on convertit les variables catégorielles en vecteurs numériques.
preprocess = ColumnTransformer(
    transformers=[
        # ('num', 'passthrough', feature_cols_num),
        ('num', StandardScaler(), feature_cols_num),
        ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
    ],
    remainder='drop',
)

## Entraînement de modèle (1) : Linear Regression

In [7]:
model_lr = Pipeline(steps=[
    ('prep', preprocess),
    ('model', LinearRegression()),
])

model_lr.fit(X_train, y_train)
pred_lr_test = model_lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, pred_lr_test)
rmse_lr = root_mean_squared_error(y_test, pred_lr_test)
r2_lr_test = r2_score(y_test, pred_lr_test)

pred_lr_train = model_lr.predict(X_train)
r2_lr_train = r2_score(y_train, pred_lr_train)

print(
    f"Linear Regression — MAE = {mae_lr:.2f}, RMSE = {rmse_lr:.2f}, "
    f"R²-test = {r2_lr_test:.3f}, R²-train = {r2_lr_train:.3f}"
)

# Export du modèle pour utilisation dans Streamlit
models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

joblib.dump(model_lr, models_dir / 'model_lr.joblib')

Linear Regression — MAE = 20.94, RMSE = 37.70, R²-test = 0.895, R²-train = 0.892


['../models/model_lr.joblib']

## Entraînement de modèle (2) : Ridge

In [8]:
model_ridge = Pipeline(steps=[
    ('prep', preprocess),
    ('model', Ridge(alpha=1.0)),
])

model_ridge.fit(X_train, y_train)
pred_ridge = model_ridge.predict(X_test)

mae_ridge = mean_absolute_error(y_test, pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, pred_ridge)
r2_ridge = r2_score(y_test, pred_ridge)

print(f"Ridge — MAE = {mae_ridge:.2f}, RMSE = {rmse_ridge:.2f}, R² = {r2_ridge:.3f}")

Ridge — MAE = 20.94, RMSE = 37.71, R² = 0.895


## Entraînement de modèle (3) : Lasso

In [9]:
model_lasso = Pipeline(steps=[
    ('prep', preprocess),
    ('model', Lasso(alpha=0.001, max_iter=10000)),
])

model_lasso.fit(X_train, y_train)
pred_lasso = model_lasso.predict(X_test)

mae_lasso = mean_absolute_error(y_test, pred_lasso)
rmse_lasso = root_mean_squared_error(y_test, pred_lasso)
r2_lasso = r2_score(y_test, pred_lasso)

print(f"Lasso — MAE = {mae_lasso:.2f}, RMSE = {rmse_lasso:.2f}, R² = {r2_lasso:.3f}")

Lasso — MAE = 20.94, RMSE = 37.71, R² = 0.895


## Entraînement de modèle (4) : Elastic Net

In [10]:
model_en = Pipeline(steps=[
    ('prep', preprocess),
    ('model', ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000)),
])

model_en.fit(X_train, y_train)
pred_en = model_en.predict(X_test)

mae_en = mean_absolute_error(y_test, pred_en)
rmse_en = root_mean_squared_error(y_test, pred_en)
r2_en = r2_score(y_test, pred_en)

print(f"ElasticNet — MAE = {mae_en:.2f}, RMSE = {rmse_en:.2f}, R² = {r2_en:.3f}")

ElasticNet — MAE = 20.93, RMSE = 37.71, R² = 0.895


## Conclusion

Observations sur le dataset de mai 2024 à juin 2025 :

Les quatre modèles affichent des performances très proches. La régression linéaire obtient
un R² de 0.895 sur le jeu de test (MAE = 20.94, RMSE = 37.70), avec un R²-train de 0.892 —
ce qui indique l'absence de surapprentissage significatif. Les modèles Ridge, Lasso et
Elastic Net n'apportent pas de gain notable, ce qui confirme que la régularisation n'est
pas nécessaire ici.

La régression linéaire simple est donc retenue et exportée comme modèle de référence pour
les étapes suivantes.